In [0]:
# CELLULE 0 — Configuration
spark.conf.set('spark.databricks.execution.timeout', 18000)
spark.conf.set("spark.sql.shuffle.partitions", "400")
print("✅ Configuration optimisée !")

In [0]:
# CELLULE 1 — Charger FT et OT
from pyspark.sql.functions import (
    col, expr, abs as spark_abs,
    row_number, unix_timestamp, when,
    broadcast, first, explode, array, lit, year
)
from pyspark.sql.window import Window

df_FT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/FT/"
)
df_OT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/OT/"
)

print(f"✅ FT : {df_FT.count():,} vols")
print(f"✅ OT : {df_OT.count():,} observations")


In [0]:

# Vérifier couverture temporelle
print("\n=== Années dans FT ===")
df_FT.select(year("FL_DATE").alias("y")).distinct().orderBy("y").show()
print("=== Années dans OT ===")
df_OT.select(col("Date").substr(1,4).alias("y")).distinct().orderBy("y").show()

# COMMAND ----------
# CELLULE 2 — Générer 13 slots horaires départ
# ✅ CORRECTION 1 — 13 observations par vol au lieu de 1
# Section 4.1 : W_o = O(A_o, t_sd), O(A_o, t_sd-1h), ..., O(A_o, t_sd-12h)

df_slots_dep = df_FT.select(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "CRS_DEP_TIME", "CRS_ELAPSED_TIME", "ARR_DELAY_NEW",
    "WEATHER_DELAY", "NAS_DELAY", "CRS_ARR_TIME",
    "DEP_DATETIME", "ARR_DATETIME",
    # Générer 13 offsets : 0h, 1h, ..., 12h
    explode(
        array([lit(i) for i in range(13)])
    ).alias("offset_h")
).withColumn(
    # Timestamp cible pour chaque slot
    "target_dep_dt",
    col("DEP_DATETIME") - expr("offset_h * INTERVAL 1 HOUR")
)

print(f"✅ Slots départ : {df_slots_dep.count():,} lignes")
print(f"   (attendu ~{df_FT.count()*13:,} = {df_FT.count():,} vols × 13)")

In [0]:
# PREMIER JOIN météo origine
# Pour chaque slot → obs. météo la plus proche dans ±30min

df_OT_origin = df_OT.select(
    col("AirportID").alias("ORIG_ID"),
    col("OBS_DATETIME").alias("OBS_ORIG_DT"),
    col("Temp").alias("orig_Temp"),
    col("Humidity").alias("orig_Humidity"),
    col("WindDirection").alias("orig_WindDir"),
    col("WindSpeed").alias("orig_WindSpeed"),
    col("Pressure").alias("orig_Pressure"),
    col("SkyCondition").alias("orig_Sky"),
    col("Visibility").alias("orig_Visibility"),
    col("WeatherType").alias("orig_WeatherType")
)

df_join1 = df_slots_dep.join(
    broadcast(df_OT_origin),
    (col("ORIGIN_AIRPORT_ID") == col("ORIG_ID")) &
    (col("OBS_ORIG_DT") >= col("target_dep_dt") - expr("INTERVAL 30 MINUTES")) &
    (col("OBS_ORIG_DT") <= col("target_dep_dt") + expr("INTERVAL 30 MINUTES")),
    "left"
).withColumn(
    "diff_orig",
    spark_abs(
        unix_timestamp("target_dep_dt") - unix_timestamp("OBS_ORIG_DT")
    )
)

# ✅ Une obs. la plus proche PAR SLOT (pas par vol !)
w_orig = Window.partitionBy(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "DEP_DATETIME", "offset_h"   # ← offset_h = clé du slot !
).orderBy("diff_orig")

df_join1 = df_join1.withColumn(
    "rn", row_number().over(w_orig)
).filter(col("rn") == 1).drop("rn", "diff_orig", "ORIG_ID", "OBS_ORIG_DT")

print(f"✅ Join1 : {df_join1.count():,} lignes")
print(f"   (attendu ~{df_FT.count()*13:,} = 1 obs. par slot)")
df_join1.show(3, truncate=False)

In [0]:
# Pivoter météo origine
# 13 lignes par vol → 1 ligne avec 13×8 colonnes météo
# orig_Temp_0h, orig_Temp_1h, ..., orig_Temp_12h
df_orig_pivot = df_join1.groupBy(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "CRS_DEP_TIME", "CRS_ELAPSED_TIME", "ARR_DELAY_NEW",
    "WEATHER_DELAY", "NAS_DELAY", "CRS_ARR_TIME",
    "DEP_DATETIME", "ARR_DATETIME"
).pivot("offset_h", list(range(13))).agg(
    first("orig_Temp").alias("Temp"),
    first("orig_Humidity").alias("Hum"),
    first("orig_WindDir").alias("WDir"),
    first("orig_WindSpeed").alias("WSpd"),
    first("orig_Pressure").alias("Pres"),
    first("orig_Sky").alias("Sky"),
    first("orig_Visibility").alias("Vis"),
    first("orig_WeatherType").alias("WType")
)

# Renommer : 0_Temp → orig_Temp_0h
renamed_orig = []
for c in df_orig_pivot.columns:
    parts = c.split("_", 1)
    if parts[0].isdigit():
        renamed_orig.append(
            col(f"`{c}`").alias(f"orig_{parts[1]}_{parts[0]}h")
        )
    else:
        renamed_orig.append(col(c))

df_orig_pivot = df_orig_pivot.select(renamed_orig)

# Sauvegarder intermédiaire
df_orig_pivot.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/join1_pivot/"
)
print(f"✅ Join1 pivot : {df_orig_pivot.count():,} vols")
print(f"📋 Colonnes   : {len(df_orig_pivot.columns)}")
print(f"   (attendu : 11 infos vol + 13×8 = 115 colonnes)")

In [0]:
# Slots horaires arrivée + DEUXIÈME JOIN
df_slots_arr = df_FT.select(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "DEP_DATETIME", "ARR_DATETIME",
    explode(
        array([lit(i) for i in range(13)])
    ).alias("offset_h")
).withColumn(
    "target_arr_dt",
    col("ARR_DATETIME") - expr("offset_h * INTERVAL 1 HOUR")
)

df_OT_dest = df_OT.select(
    col("AirportID").alias("DEST_ID"),
    col("OBS_DATETIME").alias("OBS_DEST_DT"),
    col("Temp").alias("dest_Temp"),
    col("Humidity").alias("dest_Humidity"),
    col("WindDirection").alias("dest_WindDir"),
    col("WindSpeed").alias("dest_WindSpeed"),
    col("Pressure").alias("dest_Pressure"),
    col("SkyCondition").alias("dest_Sky"),
    col("Visibility").alias("dest_Visibility"),
    col("WeatherType").alias("dest_WeatherType")
)

df_join2 = df_slots_arr.join(
    broadcast(df_OT_dest),
    (col("DEST_AIRPORT_ID") == col("DEST_ID")) &
    (col("OBS_DEST_DT") >= col("target_arr_dt") - expr("INTERVAL 30 MINUTES")) &
    (col("OBS_DEST_DT") <= col("target_arr_dt") + expr("INTERVAL 30 MINUTES")),
    "left"
).withColumn(
    "diff_dest",
    spark_abs(
        unix_timestamp("target_arr_dt") - unix_timestamp("OBS_DEST_DT")
    )
)

w_dest = Window.partitionBy(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "ARR_DATETIME", "offset_h"
).orderBy("diff_dest")

df_join2 = df_join2.withColumn(
    "rn", row_number().over(w_dest)
).filter(col("rn") == 1).drop("rn", "diff_dest", "DEST_ID", "OBS_DEST_DT")

# Pivoter météo destination
df_dest_pivot = df_join2.groupBy(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "DEP_DATETIME", "ARR_DATETIME"
).pivot("offset_h", list(range(13))).agg(
    first("dest_Temp").alias("Temp"),
    first("dest_Humidity").alias("Hum"),
    first("dest_WindDir").alias("WDir"),
    first("dest_WindSpeed").alias("WSpd"),
    first("dest_Pressure").alias("Pres"),
    first("dest_Sky").alias("Sky"),
    first("dest_Visibility").alias("Vis"),
    first("dest_WeatherType").alias("WType")
)

renamed_dest = []
for c in df_dest_pivot.columns:
    parts = c.split("_", 1)
    if parts[0].isdigit():
        renamed_dest.append(
            col(f"`{c}`").alias(f"dest_{parts[1]}_{parts[0]}h")
        )
    else:
        renamed_dest.append(col(c))

df_dest_pivot = df_dest_pivot.select(renamed_dest)

df_dest_pivot.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/join2_pivot/"
)
print(f"✅ Join2 pivot : {df_dest_pivot.count():,} vols")
print(f"📋 Colonnes   : {len(df_dest_pivot.columns)}")

In [0]:
# CELLULE 6 — Fusion finale → JT
df_orig_r = spark.read.parquet(
    "/Volumes/workspace/default/outputs/join1_pivot/"
)
df_dest_r = spark.read.parquet(
    "/Volumes/workspace/default/outputs/join2_pivot/"
)

df_JT = df_orig_r.join(
    df_dest_r,
    on=["FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
        "DEP_DATETIME", "ARR_DATETIME"],
    how="left"
).withColumn(
    "label_15", when(col("ARR_DELAY_NEW") >= 15, 1).otherwise(0)
).withColumn(
    "label_60", when(col("ARR_DELAY_NEW") >= 60, 1).otherwise(0)
)


In [0]:
# Vérification + nettoyage
# Filtrer vols sans météo

n_total   = df_JT.count()
n_no_orig = df_JT.filter(col("orig_Temp_0h").isNull()).count()
n_no_dest = df_JT.filter(col("dest_Temp_0h").isNull()).count()

print(f"Total JT avant filtre : {n_total:,}")
print(f"Sans météo origine    : {n_no_orig:,} ({n_no_orig/n_total*100:.1f}%)")
print(f"Sans météo destination: {n_no_dest:,} ({n_no_dest/n_total*100:.1f}%)")

# ✅ Garder seulement vols avec météo complète
df_JT_clean = df_JT.filter(
    col("orig_Temp_0h").isNotNull() &
    col("dest_Temp_0h").isNotNull()
)

n_clean   = df_JT_clean.count()
n_del_15  = df_JT_clean.filter(col("label_15") == 1).count()
n_del_60  = df_JT_clean.filter(col("label_60") == 1).count()

print(f"\n✅ JT après filtre météo : {n_clean:,} vols")
print(f"   ({n_clean/n_total*100:.1f}% du total)")
print(f"🏷️  Delayed 15min : {n_del_15:,} ({n_del_15/n_clean*100:.1f}%)")
print(f"🏷️  Delayed 60min : {n_del_60:,} ({n_del_60/n_clean*100:.1f}%)")
print(f"📋 Colonnes      : {len(df_JT_clean.columns)}")
print(f"   (attendu ~221 : 11 vol + 104 orig + 104 dest + 2 labels)")

# ✅ Aperçu avec bonnes colonnes
df_JT_clean.select(
    "FL_DATE", "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "ARR_DELAY_NEW",
    "orig_Temp_0h", "orig_Temp_1h", "orig_Temp_2h",
    "dest_Temp_0h", "dest_Temp_1h", "dest_Temp_2h",
    "label_15", "label_60"
).show(5, truncate=False)



In [0]:
# Sauvegarde JT finale

df_JT_clean.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/JT/"
)

print("=" * 50)
print("      RÉSUMÉ FINAL — NOTEBOOK 02")
print("=" * 50)
print(f"✈️  JT finale  : {df_JT_clean.count():,} vols")
print(f"📋 Colonnes   : {len(df_JT_clean.columns)}")
print(f"🏷️  label_15  : {df_JT_clean.filter(col('label_15')==1).count():,} delayed")
print(f"🏷️  label_60  : {df_JT_clean.filter(col('label_60')==1).count():,} delayed")
print("=" * 50)
print("✅ Notebook 02 TERMINÉ → Prêt pour Notebook 03 !")

In [0]:
# Databricks notebook source
# ================================
# NOTEBOOK 02 — DOUBLE JOIN
# Section 4.1 du papier :
# "creating Joint Table JT = {F, W_o, W_d, C}"
# ================================

# COMMAND ----------
# ================================
# CELLULE 1 — Charger FT et OT
# ================================
from pyspark.sql.functions import (
    col, expr, to_timestamp, lit,
    lpad, concat, floor
)

df_FT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/FT/"
)
df_OT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/OT/"
)

print(f"✅ FT chargée : {df_FT.count():,} vols")
print(f"✅ OT chargée : {df_OT.count():,} observations")
df_FT.show(3)
df_OT.show(3)

# COMMAND ----------
# ================================
# CELLULE 2 — PREMIER JOIN
# Section 4.1 du papier :
# W_o = météo origine de t_sd → t_sd-12h
# Clé join : (ORIGIN_AIRPORT_ID, DATE(DEP))
# ================================

# Préparer OT pour jointure origine
df_OT_origin = df_OT.select(
    col("AirportID").alias("ORIGIN_ID"),
    col("OBS_DATETIME").alias("OBS_DEP_DATETIME"),
    col("Temp").alias("orig_Temp"),
    col("Humidity").alias("orig_Humidity"),
    col("WindDirection").alias("orig_WindDir"),
    col("WindSpeed").alias("orig_WindSpeed"),
    col("Pressure").alias("orig_Pressure"),
    col("SkyCondition").alias("orig_SkyCondition"),
    col("Visibility").alias("orig_Visibility"),
    col("WeatherType").alias("orig_WeatherType")
)

# Premier Join :
# Pour chaque vol, trouver observations météo à l'origine
# dans la fenêtre [DEP_DATETIME - 12h, DEP_DATETIME]
# Le papier : "W_o = O(A_o, t_sd), O(A_o, t_sd-1h), ..., O(A_o, t_sd-12h)"

df_join1 = df_FT.join(
    df_OT_origin,
    (df_FT["ORIGIN_AIRPORT_ID"] == df_OT_origin["ORIGIN_ID"]) &
    (df_OT_origin["OBS_DEP_DATETIME"] >= 
     df_FT["DEP_DATETIME"] - expr("INTERVAL 12 HOURS")) &
    (df_OT_origin["OBS_DEP_DATETIME"] <= df_FT["DEP_DATETIME"]),
    "left"
)

print(f"✅ Après Premier Join : {df_join1.count():,} lignes")
df_join1.show(3, truncate=False)

# COMMAND ----------
# ================================
# CELLULE 3 — Garder observation
# météo la plus proche de t_sd
# Le papier : "we take the closest one
# to the weather observation time requested"
# ================================
from pyspark.sql.functions import (
    abs as spark_abs, row_number, unix_timestamp
)
from pyspark.sql.window import Window

# Calculer différence en secondes entre
# heure vol et heure observation
df_join1 = df_join1.withColumn(
    "time_diff_dep",
    spark_abs(
        unix_timestamp(col("DEP_DATETIME")) -
        unix_timestamp(col("OBS_DEP_DATETIME"))
    )
)

# Pour chaque vol, garder l'observation la plus proche
# par heure (fenêtre de 1h = 3600 secondes)
window_dep = Window.partitionBy(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "DEP_DATETIME"
).orderBy("time_diff_dep")

df_join1_closest = df_join1.withColumn(
    "rank_dep", row_number().over(window_dep)
).filter(col("rank_dep") == 1).drop("rank_dep", "time_diff_dep")

print(f"✅ Après sélection obs. plus proche : {df_join1_closest.count():,} lignes")
df_join1_closest.show(3, truncate=False)

# COMMAND ----------
# ================================
# CELLULE 4 — DEUXIÈME JOIN
# Section 4.1 du papier :
# W_d = météo destination de t_sa → t_sa-12h
# Clé join : (DEST_AIRPORT_ID, DATE(ARR))
# ================================

# Préparer OT pour jointure destination
df_OT_dest = df_OT.select(
    col("AirportID").alias("DEST_ID"),
    col("OBS_DATETIME").alias("OBS_ARR_DATETIME"),
    col("Temp").alias("dest_Temp"),
    col("Humidity").alias("dest_Humidity"),
    col("WindDirection").alias("dest_WindDir"),
    col("WindSpeed").alias("dest_WindSpeed"),
    col("Pressure").alias("dest_Pressure"),
    col("SkyCondition").alias("dest_SkyCondition"),
    col("Visibility").alias("dest_Visibility"),
    col("WeatherType").alias("dest_WeatherType")
)

# Deuxième Join :
# Pour chaque vol, trouver observations météo à destination
# dans la fenêtre [ARR_DATETIME - 12h, ARR_DATETIME]
df_join2 = df_join1_closest.join(
    df_OT_dest,
    (df_join1_closest["DEST_AIRPORT_ID"] == df_OT_dest["DEST_ID"]) &
    (df_OT_dest["OBS_ARR_DATETIME"] >=
     df_join1_closest["ARR_DATETIME"] - expr("INTERVAL 12 HOURS")) &
    (df_OT_dest["OBS_ARR_DATETIME"] <= df_join1_closest["ARR_DATETIME"]),
    "left"
)

print(f"✅ Après Deuxième Join : {df_join2.count():,} lignes")

# COMMAND ----------
# ================================
# CELLULE 5 — Garder observation
# météo la plus proche de t_sa
# ================================
df_join2 = df_join2.withColumn(
    "time_diff_arr",
    spark_abs(
        unix_timestamp(col("ARR_DATETIME")) -
        unix_timestamp(col("OBS_ARR_DATETIME"))
    )
)

window_arr = Window.partitionBy(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "DEP_DATETIME"
).orderBy("time_diff_arr")

df_JT = df_join2.withColumn(
    "rank_arr", row_number().over(window_arr)
).filter(col("rank_arr") == 1).drop("rank_arr", "time_diff_arr")

print(f"✅ Joint Table JT : {df_JT.count():,} lignes")

# COMMAND ----------
# ================================
# CELLULE 6 — Créer colonne classe C
# Section 4.2 du papier :
# C = 0 si ARR_DELAY < Th (on-time)
# C = 1 si ARR_DELAY >= Th (delayed)
# Deux seuils : 15min et 60min
# ================================
from pyspark.sql.functions import when

# Seuil 15 minutes
df_JT = df_JT.withColumn(
    "label_15",
    when(col("ARR_DELAY_NEW") >= 15, 1).otherwise(0)
)

# Seuil 60 minutes
df_JT = df_JT.withColumn(
    "label_60",
    when(col("ARR_DELAY_NEW") >= 60, 1).otherwise(0)
)

# Stats distribution classes
print("=== Distribution classes (threshold=15min) ===")
df_JT.groupBy("label_15").count().show()
# Attendu : ~80% ontime (0), ~20% delayed (1)

print("=== Distribution classes (threshold=60min) ===")
df_JT.groupBy("label_60").count().show()

# COMMAND ----------
# ================================
# CELLULE 7 — Vérification finale JT
# ================================
print("=== Colonnes JT ===")
print(df_JT.columns)

print("\n=== Aperçu JT ===")
df_JT.select(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "DEP_DATETIME",
    "ARR_DELAY_NEW",
    "orig_Temp", "orig_WindSpeed",
    "dest_Temp", "dest_WindSpeed",
    "label_15", "label_60"
).show(5, truncate=False)

# Vérifier pas de vols sans météo
n_sans_meteo_orig = df_JT.filter(
    col("orig_Temp").isNull()
).count()
n_sans_meteo_dest = df_JT.filter(
    col("dest_Temp").isNull()
).count()

print(f"⚠️  Vols sans météo origine      : {n_sans_meteo_orig:,}")
print(f"⚠️  Vols sans météo destination  : {n_sans_meteo_dest:,}")

# COMMAND ----------
# ================================
# CELLULE 8 — Sauvegarde JT
# ================================
df_JT.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/JT/"
)

print("=" * 45)
print("     RÉSUMÉ FINAL — NOTEBOOK 02")
print("=" * 45)
print(f"✈️  Joint Table JT : {df_JT.count():,} vols")
print(f"📋 Colonnes       : {len(df_JT.columns)}")
print(f"🏷️  Label 15min   : {df_JT.filter(col('label_15')==1).count():,} delayed")
print(f"🏷️  Label 60min   : {df_JT.filter(col('label_60')==1).count():,} delayed")
print("=" * 45)
print("✅ Notebook 02 TERMINÉ → Prêt pour Notebook 03 !")